In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
.appName("Retail_Sales_Performance_Dashboard") \
.getOrCreate()
spark

In [4]:
from google.colab import files

uploaded = files.upload()

Saving retail_sales.csv to retail_sales.csv


In [5]:
# LOADING DATASET
df = spark.read.csv("retail_sales.csv",header=True,inferSchema=True)
print("Total Records:", df.count())
df.printSchema()
df.show(5)

Total Records: 100000
root
 |-- sale_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sale_date: timestamp (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 |-- selling_price: integer (nullable = true)
 |-- cost_price: integer (nullable = true)
 |-- return_quantity: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- profit: integer (nullable = true)

+-------+--------+---------------+------+----------+------------+-----------+-------------------+-------------+-------------+----------+---------------+-------+-------+
|sale_id|store_id|     store_name|region|product_id|product_name|   category|          sale_date|quantity_sold|selling_price|cost_price|return_quantity|revenue| profit|
+-------+--------+---------------+------

In [7]:
# Underperforming Products
underperforming_products = df.filter((col("quantity_sold") < 10) | (col("return_quantity") > 5))
print("Underperforming Product Records")
underperforming_products.show(10)

Underperforming Product Records
+-------+--------+---------------+------+----------+------------+-----------+-------------------+-------------+-------------+----------+---------------+-------+--------+
|sale_id|store_id|     store_name|region|product_id|product_name|   category|          sale_date|quantity_sold|selling_price|cost_price|return_quantity|revenue|  profit|
+-------+--------+---------------+------+----------+------------+-----------+-------------------+-------------+-------------+----------+---------------+-------+--------+
|      3|     105|     Pune Store| South|       202|      Router|Accessories|2025-01-01 00:02:00|            4|        16684|     45482|              7|  66736| -115192|
|      8|     105|  Kolkata Store| South|       208|   Projector|Accessories|2025-01-01 00:07:00|            7|        32584|     28958|              6| 228088|   25382|
|     10|     102|    Delhi Store| South|       208|      Tablet|Accessories|2025-01-01 00:09:00|            6|       

In [8]:
# Product Summary
product_summary = (underperforming_products.groupBy("product_id","product_name").agg(sum("quantity_sold").alias("total_sales"),sum("return_quantity").alias("total_returns")))
product_summary.show()

+----------+------------+-----------+-------------+
|product_id|product_name|total_sales|total_returns|
+----------+------------+-----------+-------------+
|       208|   Projector|      10062|         1477|
|       213|  Microphone|       8293|         1243|
|       207|   Projector|       8188|         1369|
|       203|  Headphones|       9099|         1449|
|       211|      Laptop|       9583|         1518|
|       212|   Projector|       7240|         1169|
|       214|      Webcam|       8363|         1371|
|       202|      Laptop|       9241|         1439|
|       211|   Hard Disk|      10185|         1572|
|       214|   Projector|       8310|         1288|
|       203|   Hard Disk|       9369|         1342|
|       206| Smart Watch|       9426|         1468|
|       214|      Tablet|       8709|         1315|
|       213|     Monitor|       7528|         1217|
|       207|  Headphones|       9255|         1405|
|       215|       Mouse|      10560|         1598|
|       210|

In [9]:
df = df.withColumn("month",month(col("sale_date")))
df.select("sale_date","month").show(5)

+-------------------+-----+
|          sale_date|month|
+-------------------+-----+
|2025-01-01 00:00:00|    1|
|2025-01-01 00:01:00|    1|
|2025-01-01 00:02:00|    1|
|2025-01-01 00:03:00|    1|
|2025-01-01 00:04:00|    1|
+-------------------+-----+
only showing top 5 rows


In [14]:
# Monthly Revenue by Store
monthly_revenue = (df.groupBy("store_name","month").agg(round(sum("revenue"),2).alias("monthly_revenue")))
monthly_revenue.show()

+---------------+-----+---------------+
|     store_name|month|monthly_revenue|
+---------------+-----+---------------+
|Hyderabad Store|    2|     7593670891|
|     Pune Store|    1|     8450958030|
|  Chennai Store|    2|     7635706606|
|     Pune Store|    2|     7443087889|
|Bangalore Store|    2|     7668347774|
|  Kolkata Store|    2|     7679153647|
|Hyderabad Store|    1|     8588965369|
|Ahmedabad Store|    1|     8367515085|
|   Mumbai Store|    1|     8207362547|
|   Mumbai Store|    2|     7474939183|
|Bangalore Store|    1|     8480919771|
|  Kolkata Store|    1|     8285528125|
|    Delhi Store|    2|     7608500339|
|Ahmedabad Store|    2|     7458242991|
|    Delhi Store|    1|     8347739316|
|  Chennai Store|    1|     8554838345|
|Ahmedabad Store|    3|     2782019601|
|Bangalore Store|    3|     2750132182|
|    Delhi Store|    3|     2743561980|
|Hyderabad Store|    3|     2808770468|
+---------------+-----+---------------+
only showing top 20 rows


In [15]:
# Store Summary
store_summary = (monthly_revenue.groupBy("store_name").agg(round(avg("monthly_revenue"),2).alias("avg_monthly_revenue")))
store_summary.show()

+---------------+-------------------+
|     store_name|avg_monthly_revenue|
+---------------+-------------------+
|  Kolkata Store|      6.286104078E9|
|Hyderabad Store|    6.33046890933E9|
|Ahmedabad Store|      6.202592559E9|
|  Chennai Store|    6.37411388233E9|
|Bangalore Store|      6.299799909E9|
|    Delhi Store|    6.23326721167E9|
|     Pune Store|    6.24098876633E9|
|   Mumbai Store|    6.15913687933E9|
+---------------+-------------------+



In [16]:
# Save Outputs
product_summary.coalesce(1).write.mode("overwrite").option("header", True).csv("underperforming_products")
store_summary.coalesce(1).write.mode("overwrite").option("header", True).csv("store_summary")